 Configuração do Ambiente e Base de Dados


In [1]:

import pandas as pd
import numpy as np
import folium
from folium.plugins import MarkerCluster

# Gerando dados sintéticos de imóveis em Nova Iguaçu e Queimados
np.random.seed(42)
n_imoveis = 45

# Coordenadas base (Aproximadas)
# Nova Iguaçu: -22.756, -43.460
# Queimados: -22.716, -43.555

dados_imoveis = {
    'id_imovel': range(1, n_imoveis + 1),
    'cidade': np.where(np.random.rand(n_imoveis) > 0.4, 'Nova Iguaçu', 'Queimados'),
    'valor_venda': np.random.uniform(150000, 850000, n_imoveis).round(2),
    'tipo': np.random.choice(['Casa', 'Apartamento', 'Terreno'], n_imoveis)
}

df_mapa = pd.DataFrame(dados_imoveis)

# Atribuindo coordenadas com base na cidade adicionando uma pequena dispersão aleatória
def gerar_lat(cidade):
    if cidade == 'Nova Iguaçu':
        return -22.756 + np.random.uniform(-0.03, 0.03)
    return -22.716 + np.random.uniform(-0.02, 0.02)

def gerar_lon(cidade):
    if cidade == 'Nova Iguaçu':
        return -43.460 + np.random.uniform(-0.03, 0.03)
    return -43.555 + np.random.uniform(-0.02, 0.02)

df_mapa['latitude'] = df_mapa['cidade'].apply(gerar_lat)
df_mapa['longitude'] = df_mapa['cidade'].apply(gerar_lon)


Conferir:

In [2]:
df_mapa.head()

,id_imovel,cidade,valor_venda,tipo,latitude,longitude
0,1,Queimados,613765.60,Casa,-22.714426,-43.571388
1,2,Nova Iguaçu,368197.75,Terreno,-22.737554,-43.439882
2,3,Nova Iguaçu,514047.61,Apartamento,-22.732235,-43.470753
3,4,Nova Iguaçu,532697.20,Terreno,-22.766920,-43.478809
4,5,Queimados,279398.12,Casa,-22.731598,-43.573369


Criar o mapa centralizado

In [7]:
import folium

centro = [
    df_mapa['latitude'].mean(),
    df_mapa['longitude'].mean()
]

mapa_base = folium.Map(
    location=centro,
    zoom_start=12,
    tiles=None
)

for _, imovel in df_mapa.head(5).iterrows():
    popup = f"""
    Tipo: {imovel['tipo']}<br>
    Valor de venda: R$ {imovel['valor_venda']:,.2f}
    """

    folium.Marker(
        location=[
            imovel['latitude'],
            imovel['longitude']
        ],
        popup=popup
    ).add_to(mapa_base)

mapa_base

Adicionar os 5 primeiros imóveis

In [8]:
for _, imovel in df_mapa.head(5).iterrows():

    popup = f"""
    Tipo: {imovel['tipo']}<br>
    Valor de venda: R$ {imovel['valor_venda']:,.2f}
    """

    folium.Marker(
        location=[
            imovel['latitude'],
            imovel['longitude']
        ],
        popup=popup
    ).add_to(mapa_base)

mapa_base

 Marcadores circulares

In [9]:
mapa_circulos = folium.Map(
    location=centro,
    zoom_start=12,
    tiles=None
)

Adicionar os círculos

In [10]:
for _, imovel in df_mapa.iterrows():

    if imovel['cidade'] == 'Nova Iguaçu':
        cor = 'blue'
    else:
        cor = 'orange'

    folium.CircleMarker(
        location=[
            imovel['latitude'],
            imovel['longitude']
        ],
        radius=8,
        color=cor,
        fill=True,
        fill_color=cor,
        fill_opacity=0.7,
        tooltip='Clique para detalhes'
    ).add_to(mapa_circulos)

mapa_circulos

Criar o terceiro mapa

In [11]:
mapa_cluster = folium.Map(
    location=centro,
    zoom_start=12,
    tiles=None
)

cluster = MarkerCluster().add_to(mapa_cluster)

Adicionar os imóveis ao agrupamento

In [12]:
cores_tipo = {
    'Casa': 'green',
    'Apartamento': 'blue',
    'Terreno': 'gray'
}

for _, imovel in df_mapa.iterrows():

    popup = f"""
    <b>Imóvel {imovel['id_imovel']}</b><br>
    Tipo: {imovel['tipo']}<br>
    Cidade: {imovel['cidade']}<br>
    Valor de venda: R$ {imovel['valor_venda']:,.2f}
    """

    folium.Marker(
        location=[
            imovel['latitude'],
            imovel['longitude']
        ],
        popup=popup,
        icon=folium.Icon(
            color=cores_tipo[imovel['tipo']],
            icon='home',
            prefix='fa'
        )
    ).add_to(cluster)

mapa_cluster

 Salvar o mapa final

In [14]:
from google.colab import files

files.download('mapa_imoveis_baixada.html')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>